# DLAI Model Merging - Final global-TIES corrective experiment

This is the final planned experimental phase. It compares Mean, frozen Task Arithmetic, the tensor-wise TIES variant used in notebooks 03-08, and global TIES matching the authors' flatten-first implementation. All methods use fixed hyperparameters. Data subsets are fixed with `SUBSET_SEED=2026`, while training seeds vary over 7, 42, and 123. No result-dependent tuning follows this run.

No external Kaggle Input is required. Select **GPU T4 x2**, enable Internet, and choose **Save Version -> Save & Run All**.

In [ ]:
!nvidia-smi
!python --version

## Install the audited project branch

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
REPO='https://github.com/LeuxLello/Dlai-model-merging.git'
BRANCH='codex/multiseed-results'
WORKDIR=Path('/kaggle/working/Dlai-model-merging')
if WORKDIR.exists(): shutil.rmtree(WORKDIR)
subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO,str(WORKDIR)])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',str(WORKDIR)])
sys.path.insert(0,str(WORKDIR/'src')); os.chdir(WORKDIR)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Commit:',COMMIT)

## Environment, frozen protocol, and implementation checks

In [ ]:
import gc, itertools, json, platform
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForSequenceClassification
from dlai_merge.diagnostics import cosine_similarity, sign_agreement, subtract_states
from dlai_merge.evaluation import TaskEvaluator
from dlai_merge.merging import global_ties_merge, mean_merge, task_arithmetic, ties_merge
from dlai_merge.training import TrainConfig, train_specialist
assert torch.cuda.is_available(), 'Enable GPU T4 x2.'
GPU=torch.cuda.get_device_name(0); CAP=torch.cuda.get_device_capability(0)
print('GPU:',GPU,'capability:',CAP); assert CAP[0]>=7, 'Use T4 x2, not P100.'
torch.ones(1,device='cuda').add_(1)
TASKS=['sst2','imdb','mrpc','rte']; PAIRS=list(itertools.combinations(TASKS,2))
TRAIN_SEEDS=[7,42,123]; SUBSET_SEED=2026; BASE='prajjwal1/bert-mini'
TA_SCALE=0.75; TIES_DENSITY=0.2; TIES_SCALE=1.0
TRAIN_ROOT=Path('/kaggle/working/global_ties_specialists')
# A toy case proves that global and tensor-wise trimming are genuinely different.
toy_base={'small':torch.zeros(1),'large':torch.zeros(3)}
toy_a={'small':torch.tensor([2.]),'large':torch.tensor([10.,1.,.5])}
toy_b={'small':torch.tensor([1.5]),'large':torch.tensor([8.,.8,.4])}
toy_tensor=ties_merge(toy_base,[toy_a,toy_b],density=.25)
toy_global=global_ties_merge(toy_base,[toy_a,toy_b],density=.25)
assert toy_global['small'].item()==0 and toy_global['large'][0].item()==9
assert toy_tensor['small'].item()==1.75 and toy_global['small'].item()==0
print('Global-vs-tensor-wise unit check passed.')

## Train specialists and evaluate every frozen method

Each seed sees the same sampled training and evaluation examples. Only model initialization, dropout, data-loader order, and optimization randomness vary with `TRAIN_SEED`. Checkpoints are deleted after each seed; compact results are retained.

In [ ]:
specialist_rows=[]; diagnostic_rows=[]; result_rows=[]
for seed in TRAIN_SEEDS:
    print(f'\n######## TRAINING SEED {seed}; FIXED SUBSET {SUBSET_SEED} ########')
    for task in TASKS:
        summary=train_specialist(TrainConfig(
            task=task,output_root=str(TRAIN_ROOT),seed=seed,subset_seed=SUBSET_SEED,
            max_train_samples=12000,max_eval_samples=2000,epochs=3,max_steps=400,
            eval_steps=100,train_batch_size=32,eval_batch_size=64,learning_rate=2e-5))
        metric=summary['primary_metric']
        specialist_rows.append({'seed':seed,'subset_seed':SUBSET_SEED,'task':task,
          'primary_metric':metric,'score':summary['eval_metrics']['eval_'+metric]})
    enc={t:torch.load(TRAIN_ROOT/t/f'seed-{seed}'/'encoder.pt',map_location='cpu',weights_only=True) for t in TASKS}
    heads={t:torch.load(TRAIN_ROOT/t/f'seed-{seed}'/'head.pt',map_location='cpu',weights_only=True) for t in TASKS}
    base_model=AutoModelForSequenceClassification.from_pretrained(BASE,num_labels=2)
    base={k:v.detach().cpu().clone() for k,v in base_model.base_model.state_dict().items()}
    del base_model; gc.collect(); torch.cuda.empty_cache()
    evaluators={t:TaskEvaluator(t,heads[t],seed=seed,subset_seed=SUBSET_SEED,
      max_eval_samples=2000,output_root=f'/kaggle/working/global-ties-eval-{seed}') for t in TASKS}
    references={}
    for task in TASKS:
        score=evaluators[task].evaluate(enc[task]); references[task]=score['primary_score']
    for left,right in PAIRS:
        va=subtract_states(enc[left],base); vb=subtract_states(enc[right],base)
        diagnostic_rows.append({'seed':seed,'subset_seed':SUBSET_SEED,'pair':f'{left}+{right}',
          'cosine_similarity':cosine_similarity(va,vb),'sign_agreement':sign_agreement(va,vb)})
        candidates={
          'mean':mean_merge(base,[enc[left],enc[right]]),
          'task_arithmetic':task_arithmetic(base,[enc[left],enc[right]],scale=TA_SCALE),
          'tensorwise_ties':ties_merge(base,[enc[left],enc[right]],density=TIES_DENSITY,scale=TIES_SCALE),
          'global_ties':global_ties_merge(base,[enc[left],enc[right]],density=TIES_DENSITY,scale=TIES_SCALE)}
        for method,state in candidates.items():
            for task in (left,right):
                score=evaluators[task].evaluate(state)['primary_score']
                result_rows.append({'seed':seed,'subset_seed':SUBSET_SEED,'pair':f'{left}+{right}',
                  'task':task,'method':method,'score':score,'specialist_score':references[task],
                  'retained':score/references[task]})
        print(seed,left,right,'done')
    del evaluators,heads,enc,base; gc.collect(); torch.cuda.empty_cache()
    for task in TASKS: shutil.rmtree(TRAIN_ROOT/task/f'seed-{seed}',ignore_errors=True)

## Frozen analysis and final verdict

In [ ]:
results=pd.DataFrame(result_rows); specialists=pd.DataFrame(specialist_rows); diagnostics=pd.DataFrame(diagnostic_rows)
per_pair=(results.groupby(['seed','pair','method'],as_index=False)
  .agg(mean_retained=('retained','mean'),worst_retained=('retained','min')))
method_summary=(per_pair.groupby('method',as_index=False)
  .agg(mean_retained=('mean_retained','mean'),sd=('mean_retained','std'),
       worst_retained=('worst_retained','min')))
a=per_pair[per_pair.method=='global_ties'][['seed','pair','mean_retained']].rename(columns={'mean_retained':'global_ties'})
b=per_pair[per_pair.method=='tensorwise_ties'][['seed','pair','mean_retained']].rename(columns={'mean_retained':'tensorwise_ties'})
paired=a.merge(b,on=['seed','pair']); paired['delta']=paired.global_ties-paired.tensorwise_ties
rng=np.random.default_rng(2026); values=paired.delta.to_numpy()
boot=np.array([rng.choice(values,size=len(values),replace=True).mean() for _ in range(10000)])
comparison=pd.DataFrame([{'n_pair_seed_units':len(values),'mean_delta':values.mean(),
 'median_delta':np.median(values),'win_rate':(values>0).mean(),'tie_rate':np.isclose(values,0).mean(),
 'worst_delta':values.min(),'best_delta':values.max(),
 'bootstrap_ci_low':np.quantile(boot,.025),'bootstrap_ci_high':np.quantile(boot,.975)}])
display(method_summary); display(comparison); display(paired.sort_values('delta'))

## Figures and reproducible output bundle

In [ ]:
OUT=Path('/kaggle/working/global_ties_corrective_results'); OUT.mkdir(exist_ok=True)
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
sns.barplot(data=per_pair,x='method',y='mean_retained',errorbar='sd',ax=axes[0])
axes[0].axhline(1,color='grey',ls='--'); axes[0].tick_params(axis='x',rotation=20); axes[0].set_title('Fixed-subset three-seed comparison')
sns.stripplot(data=paired,x='pair',y='delta',hue='seed',dodge=True,ax=axes[1])
axes[1].axhline(0,color='black',lw=1); axes[1].tick_params(axis='x',rotation=25); axes[1].set_title('Global minus tensor-wise TIES')
fig.tight_layout(); fig.savefig(OUT/'global_ties_corrective.png',dpi=180,bbox_inches='tight'); plt.show()
results.to_csv(OUT/'all_task_results.csv',index=False); specialists.to_csv(OUT/'specialists.csv',index=False)
diagnostics.to_csv(OUT/'diagnostics.csv',index=False); per_pair.to_csv(OUT/'per_seed_pair.csv',index=False)
method_summary.to_csv(OUT/'method_summary.csv',index=False); paired.to_csv(OUT/'paired_ties_comparison.csv',index=False)
comparison.to_csv(OUT/'global_vs_tensorwise_summary.csv',index=False)
metadata={'purpose':'final corrective global-TIES comparison','commit':COMMIT,'base_model':BASE,
 'tasks':TASKS,'training_seeds':TRAIN_SEEDS,'subset_seed':SUBSET_SEED,'max_steps':400,
 'task_arithmetic_scale':TA_SCALE,'ties_density':TIES_DENSITY,'ties_scale':TIES_SCALE,
 'gpu':GPU,'python':platform.python_version(),'torch':torch.__version__,
 'confirmatory_no_further_tuning':True,'bootstrap_resamples':10000}
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
archive=shutil.make_archive('/kaggle/working/global_ties_corrective_results','zip',OUT)
print('DOWNLOAD THIS FILE:',archive)